In [1]:
import pandas as pd

# あなたが作ったCSVを読み込む
df = pd.read_csv("npb_2023_main_fourth_batter_stats.csv")

# 数値化
num_cols = ["本塁打", "OBP", "SLG", "OPS", "打点", "試合", "打席", "打数", "安打", "四球", "死球", "三振", "併殺打"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# zスコア標準化
def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

df["z_HR"] = zscore(df["本塁打"])
df["z_OBP"] = zscore(df["OBP"])
df["z_SLG"] = zscore(df["SLG"])
df["z_OPS"] = zscore(df["OPS"])

# スコア作成
df["長打型スコア"] = 0.5 * df["z_HR"] + 0.3 * df["z_SLG"] + 0.2 * df["z_OPS"]
df["総合型スコア"] = 0.4 * df["z_OBP"] + 0.3 * df["z_SLG"] + 0.3 * df["z_OPS"]

# 分類
df["4番タイプ"] = df.apply(
    lambda row: "長打型" if row["長打型スコア"] > row["総合型スコア"] else "総合型",
    axis=1
)

# 見やすい形に整理
result = df[[
    "球団", "選手名", "4番出場回数",
    "本塁打", "OBP", "SLG", "OPS", "打点",
    "長打型スコア", "総合型スコア", "4番タイプ"
]].copy()

# スコアを見やすく丸める
result["長打型スコア"] = result["長打型スコア"].round(3)
result["総合型スコア"] = result["総合型スコア"].round(3)

# 保存
result.to_csv("npb_2023_main_fourth_batter_types.csv", index=False, encoding="utf-8-sig")

print(result.sort_values("長打型スコア", ascending=False))
print("\n保存完了: npb_2023_main_fourth_batter_types.csv")

        球団     選手名  4番出場回数  本塁打    OBP    SLG    OPS   打点  長打型スコア  総合型スコア  \
6       巨人   岡本 和真     140   41  0.374  0.584  0.958   93   1.890   1.297   
0     DeNA    牧 秀悟     143   29  0.337  0.530  0.867  103   0.781   0.341   
3     ヤクルト   村上 宗隆     139   31  0.375  0.500  0.875   84   0.764   0.634   
8       楽天   浅村 栄斗     119   26  0.368  0.462  0.829   78   0.210   0.226   
2   ソフトバンク   柳田 悠岐      74   22  0.378  0.484  0.861   85   0.168   0.544   
1    オリックス    森 友哉      71   18  0.385  0.508  0.893   64   0.136   0.839   
4      ロッテ    ポランコ      88   26  0.312  0.450  0.762   75   0.006  -0.656   
10      阪神   大山 悠輔     143   19  0.403  0.456  0.859   78  -0.133   0.674   
9       西武    マキノン      47   15  0.327  0.401  0.728   50  -0.900  -0.840   
5       中日   石川 昂弥      85   13  0.282  0.394  0.676   45  -1.156  -1.530   
7       広島  マクブルーム      50    6  0.305  0.354  0.659   31  -1.764  -1.529   

   4番タイプ  
6    長打型  
0    長打型  
3    長打型  
8    総合型  
2    総合型  
1    総合型 